# Researcher: [Molderon](https://github.com/Molderon)
# Machine - Puritan Bennett 840 <br>
> - Contains origanic waveform bio-signals form artificial ling augmentation. <br>
> - Singal timestamp = 50hz | .csv contains ~350h of continuous pressure and flow readings. <br>
> - Data is streamed via 5 individual patients, identity is non-reconstructable.  <br>
> - Operative Mode: invecive
# Source
> University of California Davis Medical Center | [link](https://health.ucdavis.edu/medical-center/) <br>

In [ ]:
#--<Python Standard Libs>--
import time, warnings, joblib
import sys, os, subprocess
from dataclasses import dataclass, field
from typing import List, Tuple

# --<DataScience>--
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from scipy.spatial import ConvexHull, QhullError
from scipy.fft import rfft
from scipy.spatial.qhull import QhullError
from scipy.fft import fft

from pandas.plotting import scatter_matrix
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

# --<Feature Engineering>--
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.interpolate import interp1d
from scipy.interpolate import BSpline, splrep


# --<Machine Learning>--
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

assert sys.version_info >= (3, 5)

# --<Plotting Style>--
plt.style.use('dark_background')
csfont = {'fontname':'Comic Sans MS'}
hfont = {'fontname':'Helvetica'}

# --<ignore>--
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
warnings.filterwarnings("ignore")

In [13]:
def Import_Dataset(ds_name: str, trunk_target: bool) -> pd.DataFrame:
    dataset = pd.read_csv(ds_name)
    if trunk_target:
        dataset = dataset.drop(['Target_Class_0', 'Target_Class_1', 'Target_Class_2',
                                'Target_Class_3', 'Target_Class_4'], axis=1)
        dataset.info()
    return dataset

# Objectives
> - 1. Excract a un-supperviced dataset.
> - 2. Excract a sub-set describing geometrical properties.
> - 3. Perform Generic exploratories
> - 4. Wavelet Transform

In [ ]:
dataset = Import_Dataset("supperviced_PAV.csv", trunk_target= True)
dataset.to_csv("Organic_InvasivePAV.csv")
dataset = Import_Dataset("supperviced_PAV.csv", trunk_target= False)
dataset.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7775200 entries, 0 to 7775199
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   Breath_ID  int64  
 1   Time_Step  int64  
 2   Flow       float64
 3   Pressure   float64
dtypes: float64(2), int64(2)
memory usage: 237.3 MB


### Volume Calculation from Flow

Volume ($V$) is the integral of Flow ($\dot{V}$) over time ($t$).

$$V(t) = \int_{0}^{t} \dot{V}(\tau) d\tau$$

For discrete data with a constant time step ($\Delta t$), the volume at the $k^{th}$ point ($V_k$) is calculated as:

$$V_k = \left( \sum_{i=1}^{k} \dot{V}_i \right) \cdot \Delta t$$

In [26]:
time_step: float = 0.02

def Terraform_DataSet(group: pd.DataFrame, time_step: float):
    features = {}
    Breath_ID = group['Breath_ID'].iloc[0]
    
    Flow, Pressure = "Flow", "Pressure"
    
    mask = group[[Flow, Pressure]].notna().all(axis=1)
    valid_group = group.loc[mask].reset_index(drop=True)
    
    if len(valid_group) < 2:
        return pd.DataFrame([features]).assign(Breath_ID=Breath_ID)
    
    Flow_array = valid_group[Flow].to_numpy() 
    P = valid_group[Pressure].to_numpy()

    # 1. Volume Calculation (Flow Integration using 0.02s step)
    # Volume (TV) is the cumulative integral of Flow over time.
    TV_cumsum = np.cumsum(Flow_array) * time_step
    
    # Standardize Volume: set the lowest point (EEV) to 0.
    TV_min_actual = TV_cumsum.min()
    TV = TV_cumsum - TV_min_actual 
    
    
    # --- Geometrical Feature Extraction ---
    
    # 2. Area (Work of Breathing)
    features["Polynomial_Area"] = np.abs(np.trapz(P, TV))
    
    # 3. Convex Hull Area
    if len(TV) >= 3:
        try:
            hull = ConvexHull(np.column_stack((TV, P)))
            features["Hull_Area"] = hull.volume
        except QhullError:
            features["Hull_Area"] = 0.0
    else:
        features["Hull_Area"] = 0.0
    
    # 4. Signal Frequency (FFT)
    for name, arr in [('TV', TV), ('P', P)]:
        ft = np.abs(rfft(arr))[:3]
        for i, val in enumerate(ft, 1):
            features[f"ft_{name}_{i}"] = val
    
    # 5. Extremes, Perimeter, Centroid, and Ranges
    diffs = np.diff(np.column_stack((TV, P)), axis=0)
    
    # Missing Feature: Extremes
    features["TV_Max"] = TV.max()   
    features["TV_Min"] = TV.min()   
    features["P_Max"] = P.max()     
    features["P_Min"] = P.min()     
    
    # Standard Range Features
    features.update({
        "Perimeter": np.sum(np.hypot(diffs[:, 0], diffs[:, 1])),
        "Width_TV_Range": np.ptp(TV),    # Tidal Volume (TV) Range
        "Height_P_Range": np.ptp(P),    # Driving Pressure (P_Max - P_Min)
        "Centroid_X_TV": TV.mean(),
        "Centroid_Y_P": P.mean(),
    })
    
    # Missing Feature: Compliance and Ratio
    P_range = features["Height_P_Range"]
    TV_range = features["Width_TV_Range"]
    
    # Dynamic Compliance (Volume Range / Pressure Range)
    features["Dynamic_Compliance"] = TV_range / P_range if P_range != 0 else 0.0
    # Aspect Ratio (Height / Width)
    features["Aspect_Ratio_H_W"] = P_range / TV_range if TV_range != 0 else 0.0
    
    # 6. Curvature Calculation
    dx = np.gradient(TV)
    dy = np.gradient(P)
    ddx, ddy = np.gradient(dx), np.gradient(dy)
    with np.errstate(divide='ignore', invalid='ignore'):
        curvature_val = np.nanmean(np.abs(dx*ddy - dy*ddx) / (dx**2 + dy**2)**1.5)
    features["Curvature"] = curvature_val if not np.isnan(curvature_val) else 0.0

    return pd.DataFrame([features]).assign(Breath_ID=Breath_ID)


def Feature_Engineering(df: pd.DataFrame) -> pd.DataFrame:
    # Pass the determined time step (0.02s) to the data transformation function
    features = pd.concat(
        [Terraform_DataSet(g, time_step) for _, g in df.groupby('Breath_ID', sort=False)],
        ignore_index=True
    )
    return df.merge(features, on='Breath_ID', how='left')

In [ ]:
dataset = Feature_Engineering(dataset)
dataset = dataset.groupby('Breath_ID', as_index=False).first()
dataset.to_csv("Extr_feature.csv")
dataset.head()

,Breath_ID,Time_Step,Flow,Pressure,Target_Class_0,Target_Class_1,Target_Class_2,Target_Class_3,Target_Class_4,Polynomial_Area,...,P_Max,P_Min,Perimeter,Width_TV_Range,Height_P_Range,Centroid_X_TV,Centroid_Y_P,Dynamic_Compliance,Aspect_Ratio_H_W,Curvature
0,0,0,2.24,5.14,0,1,0,0,0,288.263816,...,35.11,0.0,109.729958,20.8014,35.11,3.435586,2.374950,0.592464,1.687867,3.759296
1,1,0,2.14,5.22,0,0,0,0,0,314.842590,...,36.81,0.0,139.813347,21.7208,36.81,1.917915,2.650138,0.590079,1.694689,17.825141
2,2,0,2.23,4.83,0,1,0,0,0,400.419550,...,35.00,0.0,164.485692,24.8204,35.00,5.208114,2.680025,0.709154,1.410130,4.326186
3,3,0,2.12,5.15,1,0,0,0,0,57.091896,...,40.01,0.0,85.336416,9.0620,40.01,5.346637,0.627500,0.226493,4.415140,2.060282
4,4,0,3.41,1.28,1,0,0,0,0,293.877274,...,36.23,0.0,123.345407,19.2414,36.23,9.440504,2.214587,0.531090,1.882919,2.221511
